In [5]:
import os
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

LABEL_MAP = {
    "normal": 0, "crack": 1, "leak": 2, "efflorescence": 3,
    "detachment": 4, "reticular crack": 5, "spalling": 6,
    "material separation": 7, "rebar": 8, "damage": 9, "exhilaration": 10
}

def generate_validation_masks(json_dir, output_mask_dir):
    os.makedirs(output_mask_dir, exist_ok=True)
    
    mask_count = 0
    normal_count = 0
    defect_count = 0
    
    json_paths = list(Path(json_dir).rglob("*.json"))
    
    for json_path in tqdm(json_paths, desc="마스크 생성"):
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
                
        img_info = data.get("image", {})
        
        resolution = img_info.get("resolution", [1920, 1080])
        w, h = resolution[0], resolution[1]
        
        mask = np.zeros((h, w), dtype=np.uint8)
        
        is_defect = img_info.get("object_included", "N")
        
        if is_defect == "Y":
            annotations = img_info.get("annotations", [])
            drawn_shapes = 0
            
            for ann in annotations:
                label_str = ann.get("label", "").lower().strip().replace("_", " ")
                
                if label_str in LABEL_MAP:
                    class_id = LABEL_MAP[label_str]
                    
                    coords = ann.get("coordinates") or ann.get("points") or ann.get("polygon") or ann.get("data") or []
                    if not coords: continue
                    
                    pts = np.array(coords, np.float32).astype(np.int32)
                    if len(pts.shape) == 1:
                        pts = pts.reshape(-1, 2)
                        
                    shape_type = str(ann.get("shape") or ann.get("type", "Polygon")).lower()
                    
                    if "polyline" in shape_type or "line" in shape_type:
                        cv2.polylines(mask, [pts], isClosed=False, color=class_id, thickness=5)
                    else:
                        cv2.fillPoly(mask, [pts], color=class_id)
                        
                    drawn_shapes += 1
                    
            if drawn_shapes > 0:
                defect_count += 1
            else:
                normal_count += 1
        else:
            normal_count += 1 
            
        original_img_name = img_info.get("name", json_path.stem + ".jpg")
        mask_filename = Path(original_img_name).stem + ".png"
        mask_save_path = os.path.join(output_mask_dir, mask_filename)
        
        is_success, encoded_mask = cv2.imencode('.png', mask)
        if is_success:
            encoded_mask.tofile(mask_save_path)
            
        mask_count += 1

    print(f" 결함: {defect_count:,}장")
    print(f" 정상: {normal_count:,}장")

VAL_JSON_DIR = r"D:\Study\HumanStudy\Dataset\Validation\02.라벨링데이터"
VAL_MASK_SAVE_DIR = r"D:\Study\HumanStudy\Dataset\Validation_Masks_2"

generate_validation_masks(VAL_JSON_DIR, VAL_MASK_SAVE_DIR)

마스크 생성: 100%|███████████████████████████████████████████████████████████████| 52500/52500 [10:04<00:00, 86.85it/s]


 결함: 50,000장
 정상: 2,500장
